# 04 Generalization: per-attack EER on ASVspoof eval

Run only after `hypotheses.md` was frozen (commit 1d896cd). Each final model is scored on eval once.
Per-attack EER = that attack's fakes against all eval real clips (`deepvoice.metrics.per_attack_eer`).
A16 and A19 reuse training algorithms A04 and A06 ("repeated"); the other 11 are unseen.

**Rung 0** (silence/duration only). The rung 0 model is the one with the lower dev EER: the
depth-3 tree (dev 28.0% vs 30.4% for logistic regression). It is refit here deterministically
(seed 0, identical to `results/rung0.csv`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer
from deepvoice.models import build

p = pd.read_parquet("data/protocols.parquet")
S = {s: pd.read_parquet(f"data/features/silence_{s}.parquet").merge(p[["file", "label", "speaker", "attack"]], on="file")
     for s in ["train", "eval"]}
F = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
rows = {}
for m, params in [("tree", {"max_depth": 3}), ("logreg", {"C": 1.0})]:
    model = build(m, 0, **params).fit(S["train"][F].to_numpy(), S["train"].label.to_numpy())
    s = model.predict_proba(S["eval"][F].to_numpy())[:, 1]
    rows[m] = per_attack_eer(S["eval"].label, s, S["eval"].attack)
pa = pd.DataFrame(rows).mul(100)
pa["type"] = ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]
pa.to_csv(ROOT / "results/rung0_eval_per_attack.csv", float_format="%.2f")
r0 = pd.read_csv(ROOT / "results/rung0.csv").query("split == 'eval'").set_index("model")
pa.round(1)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.2))
d = pa["tree"]
x = range(len(d))
bars = ax.bar(x, d, color=["#c3c2b7" if t == "repeated" else "#52514e" for t in pa.type], width=0.7)
for b, t in zip(bars, pa.type):
    if t == "repeated":
        b.set_hatch("////"); b.set_edgecolor("white")
for i, v in zip(x, d):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.4, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.axhline(100 * r0.loc["tree", "eer"], color="#e34948", ls="--", lw=1.2)
ax.text(-0.4, 100 * r0.loc["tree", "eer"] + 1.5, f"pooled eval EER {100 * r0.loc['tree', 'eer']:.1f}%", fontsize=8, color="#e34948")
ax.set_xticks(list(x), d.index); ax.set_ylim(0, 60)
ax.set_ylabel("eval EER % (silence/duration only)")
from matplotlib.patches import Patch

ax.legend(handles=[Patch(color="#52514e", label="unseen attack"), Patch(facecolor="#c3c2b7", hatch="////", edgecolor="white", label="repeated (A16=A04, A19=A06)")],
          frameon=False, loc="upper left", bbox_to_anchor=(0, 0.72), fontsize=8)
ax.set_title("Rung 0 on ASVspoof eval, per attack", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.png", dpi=200)
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.pdf")